<a href="https://colab.research.google.com/github/cuiandrew08-lab/LiDARFusionLearning/blob/main/HAN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
import numpy as np

from google.colab import drive
drive.mount("/content/drive", force_remount = False)

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision.transforms as transforms
from tqdm.notebook import tqdm

import random

from torch.utils.data import Dataset, DataLoader

#import tensorflow as tf

TORCH_version = torch.__version__.split('+')[0]
CUDA_version = '128'

#!pip install torch-scatter torch-sparse torch-cluster -f https://data.pyg.org/whl/torch-{TORCH_version}+cu{CUDA_version}.html

#import torch_sparse

!pip install torch-geometric
import torch_geometric

#from scipy.ndimage import maximum_filter
#from scipy.spatial._qhull import ConvexHull

import sys

import matplotlib.pyplot as plt

Mounted at /content/drive


In [4]:
class MetaPathPassing(nn.Module):

  def __init__(self, in_channels, out_channels, dropout = 0.2):

    super().__init__()
    self.GAT = torch_geometric.nn.conv.GATv2Conv(in_channels, out_channels, heads = 4, dropout = 0.2)


In [ ]:
class HAN(nn.Module):

  def __init__(self, metapaths, in_channels, hidden_channels, out_channels, dropout = 0.2):

    super().__init__()
    self.metapaths = list(metapaths.keys())
    self.MetaPathNetworks = nn.ModuleDict({path: torch_geometric.nn.conv.GATv2Conv(in_channels, hidden_channels[0], heads = 4, dropout = 0.2)
                                           for path, (d_src, d_dst) in metapaths})
    self.SemanticNetwork = nn.Sequential(nn.Linear(hidden_channels[0],hidden_channels[1]), nn.Tanh(), nn.Linear(hidden_channels[1], 1, bias = False))

  def forward(self, x: dict, edge_indices: dict): #both are dictionaries, keys corresponding to each metapath

    Z_m = [F.elu(self.MetaPathNetworks[mp](x[mp], edge_indices[mp])) for mp in self.metapaths] #each [N, D]
    Z_m = torch.stack(Z_m, dim = 1) #(N, M, D)
    w_m = self.SematicNetwork(Z_m).mean(dim=0) #semanticnet maps all nodes individaully, then take mean for semantic level embedding
    beta_m = F.softmax(w_m, dim = 0)
    out = (beta_m.unsqueeze(0) * Z_m).sum(dim =1) #sum over all metapaths

    return out, beta_m.squeeze()

